In [ ]:
from lib.file_func import get_instrument_type
from lib.peak import fit_n_peaks
import numpy as np
from lib.file_func import load_file
from lib.file_func import get_instrument_type
import asyncio
import os
from concurrent.futures import ProcessPoolExecutor
import json
import plotly.graph_objects as go

In [ ]:
def segment_spec(sum_spec):
    """Perform segmentation of Orbitrap spectrum

    :param sum_spec: sum of spectra along time dimension
    :type sum_spec: array-like
    :return: list of segment indices
    :rtype: list
    """
    # Get non-zero indices
    non_zero_indices = np.flatnonzero(sum_spec)
    # Split in chunks taking into account repeating zeros
    non_zero_indices = np.split(
        non_zero_indices, np.where(np.diff(non_zero_indices) > 2)[0] + 1
    )
    # Add zeros on chunk borders
    non_zero_indices = [
        np.concatenate(([chunk[0] - 1], chunk, [chunk[-1] + 1]))
        for chunk in non_zero_indices
    ]
    # Check wrong indices on the spectrum ends
    if non_zero_indices[0][0] < 0:
        # Remove negative index in the first chunk
        non_zero_indices[0] = non_zero_indices[0][1:]
    if non_zero_indices[-1][-1] == len(sum_spec):
        # Remove out-of-range index from the last chunk
        non_zero_indices[-1][-1] = non_zero_indices[-1][:-1]
    # Remove short chunks
    non_zero_indices = [chunk for chunk in non_zero_indices if len(chunk) > 4]
    return non_zero_indices

In [ ]:
def resolution_function(mz):
    return 1.725e6 / np.sqrt(mz)


with open("C:/Users/alvai/Desktop/peak_shape.json", "r") as f:
    peak_shape = json.load(f)

In [ ]:
# Load file
file_name = "KORBI2_20240328_1743_MION2_DBrMe_HCldil_HSI_100_Clean_420_40-500mz"

instrument_type = get_instrument_type(file_name)
f = load_file(file_name)

time_stamps = f.signal.time.values
print(f"Number of time sweeps is {len(f.signal.time)}")

In [ ]:
async def fit_time_sweep(file, time_index=0, fit_sum=False):
    u_list = np.arange(10, 600)

    if fit_sum == True:
        spec = file.signal.interpolate_na(dim="mz", method="linear").sum(dim="time")
    else:
        spec = file.signal.sel(time=time_stamps[time_index]).dropna(dim="mz", how="all")

    mz = spec.mz

    if instrument_type == "orbi":
        # Stack mass ranges
        u_range = np.vstack([u_list - 0.5, u_list + 0.5])
        # Broadcast the u_range array to have the same shape as mz
        u_range = u_range[:, :, np.newaxis]
        # Create boolean masks indicating which elements of spec fall within each range
        mask_u_list = (mz.values >= u_range[0]) & (mz.values <= u_range[1])
        mask_u_list = mask_u_list.any(axis=0)
        # Update mz and spec
        mz = mz.values[mask_u_list]
        spec = spec.values[mask_u_list]

        if spec.size == 0:
            # Nothing to fit
            specs_to_fit = []
        else:
            # Get mz/spectrum pairs to fit from segmented spectrum
            specs_to_fit = [(mz[chunk], spec[chunk]) for chunk in segment_spec(spec)]

    cpu_cores = os.cpu_count()
    max_workers = max(1, cpu_cores // 2)
    executor = ProcessPoolExecutor(max_workers=max_workers)
    loop = asyncio.get_event_loop()

    # Fill in asynchronous operations
    futures = [
        loop.run_in_executor(
            executor,
            fit_n_peaks,
            mz_to_fit,
            spec_to_fit,
            peak_shape,
            resolution_function(mz_to_fit.mean()),
            0.8,
            5,
        )
        for mz_to_fit, spec_to_fit in specs_to_fit
    ]

    new_peaks = []
    for i, future in enumerate(asyncio.as_completed(futures)):
        fit, peaks = await future
        if fit:
            new_peaks.extend(peaks)
        print(f"Peak detection progress: {(i+1)/len(futures):.2f}")
    executor.shutdown()

    if len(new_peaks) > 0:
        new_peak_mzs = list(zip(*new_peaks))[0]
        new_peak_heights = list(zip(*new_peaks))[1]
        new_peak_areas = list(zip(*new_peaks))[3]

    return new_peak_mzs, new_peak_heights, new_peak_areas

In [ ]:
fitted_time_sweeps = []
for i, val in enumerate(time_stamps):
    fitted_time_sweeps.append(await fit_time_sweep(f, time_index=1))

In [ ]:
sum_peak_mzs, sum_peak_heights, sum_peak_areas = await fit_time_sweep(f, fit_sum=True)

In [ ]:
print("Nans in sum: ", np.isnan(f.signal.sum(dim="time").mz).sum())
print("Nans in mean: ", np.isnan(f.signal.mean(dim="time").mz).sum())

In [ ]:
sum_spec = f.signal.sum(dim="time")  # .dropna(dim="mz", how="all")
mean_spec = f.signal.mean(dim="time")  # .dropna(dim="mz", how="all")
short_spec = f.signal.sel(mz=slice(143.91, 143.915))

mean_spec.sel(mz=slice(143.91, 143.915)).values, len(
    mean_spec.sel(mz=slice(143.91, 143.915)).values
)

In [ ]:
import plotly.colors as colors

num_colors = len(time_stamps)
color_scale = colors.qualitative.Plotly

color_list = color_scale[:num_colors]


time_index = 5

# spec = f.signal.sel(time=time_stamps[time_index]).dropna(dim="mz", how="all")
sum_spec = f.signal.interpolate_na(dim="mz", method="linear").sum(
    dim="time"
)  # .dropna(dim="mz", how="all")
mean_spec = f.signal.mean(dim="time").dropna(dim="mz", how="all")

fig = go.Figure(
    go.Scatter(
        x=sum_spec.mz,
        y=sum_spec.values / len(time_stamps),
        name="Averaged sum of all spectra",
    )
)

fig.add_trace(
    go.Scatter(
        x=mean_spec.mz,
        y=mean_spec,
        name="Mean of all spectra",
        line=dict(dash="dash"),
    )
)

for ind, sweep in enumerate(fitted_time_sweeps):
    fig.add_trace(
        go.Scatter(
            x=sweep[0],
            y=sweep[1],
            mode="markers",
            marker=dict(color=color_list[ind]),
            name=f"Time sweep {ind}",
        )
    )
    # for peak_mz, peak_height, _ in zip(*sweep):
    #     fig.add_vline(x=peak_mz, line_width=2, line_color="black", y0=0, y1=peak_height)

# for i, val in enumerate(time_stamps):
#     fig.add_trace(
#         go.Scatter(
#             x=f.signal.sel(time=time_stamps[i]).dropna(dim="mz", how="all").mz,
#             y=f.signal.sel(time=time_stamps[i]).dropna(dim="mz", how="all"),
#         )
#     )

fig.show()